In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact
import numpy as np
import warnings
import statistics as stt
#import bamboolib as bam
#from brokenaxes import brokenaxes
import os
#import plotly.express as px

In [5]:
df = pd.read_csv('329_U copy.exp')
df

,Neptune Analysis Data Report\t\t\t\t\t\t\t\t\t\t\t
0,Filename: C:\Neptune\User\Neptune\Data\Boron\2...
1,Date: 16/02/2022\t\t\t\t\t\t\t\t\t\t\t
2,Sample ID: \t\t\t\t\t\t\t\t\t\t\t
3,Method Name: C:\Neptune\User\Neptune\Methods\B...
4,Wheel ID: \t\t\t\t\t\t\t\t\t\t\t
...,...
228,***\tStdDev (%)\t1.2031347157688673e+001\t9.34...
229,***\tStdDev (abs)\t3.7223311727155549e-005\t1....
230,***\tValid Values\t200(200)\t200(200)\t200(200...
231,***\tExtErr\tNOT AVAILABLE\tNOT AVAILABLE\tNOT...


#  1. load all .exp files

In [9]:
def dirFiles(fDir, ending):
    fNames = []
    for i in os.listdir(fDir):
        if i.endswith(ending):
            fNames.append(i)
    return fNames

dataFiles = dirFiles('data', 'csv')

# 2. Define a function for single data

In [23]:
def outlierTest(measurement, factorSD):
    element_signal = np.array(measurement)
    mean = np.mean(element_signal, axis=0)
    sd = np.std(element_signal, axis=0)
    return [x for x in measurement if (x > mean - factorSD * sd) or (x < mean + factorSD * sd)]
    

def calculateSingleFile(filename, bg_10B, factorSD):
#upload data:
    df = pd.read_csv('data/' + filename, header=22)
#substract backgroud:
    endnum = df['10B'][df['10B'] == 'L3'].index[0]
    B_10 = []
    fils = []
    filw = []
    for i in df['10B'][0:endnum]:
        resi = float(i)     #convert everything from str to float into the random variable resi
        B_10.append(resi)
    for i in B_10:
        if (i > bg_10B) == True:
            fils.append(i)
        else:
            filw.append(i)

    outlier_corrected_signal =outlierTest(fils, factorSD)
    outlier_corrected_backg =outlierTest(filw, factorSD)

    B_10_aver = sum(outlier_corrected_signal )/len(outlier_corrected_signal )
    B_10_backg_aver = sum(outlier_corrected_backg)/len(outlier_corrected_backg)
    
    return(B_10_aver, B_10_backg_aver)

# 3. loop 1

In [24]:
res = []
for i in dataFiles:
    res.append(calculateSingleFile(i, 0.0005, 2))

In [25]:
res

[(0.0006765908562486906, 0.000441771933376136),
 (0.0006765999999999999, 0.0004417368421052631),
 (0.0046733396226415124, 0.00044155319148936177)]

### Some old code

In [ ]:
#check for outliers for signal:
    element_signal = np.array(fils)
    mean = np.mean(element_signal, axis=0)
    sd = np.std(element_signal, axis=0)
    final_list_signal = [x for x in fils if (x > mean - factorSD * sd)]
    final_list_signal = [x for x in final_list_signal if (x < mean + factorSD * sd)]
    
#check for outliers for background:
    element_backg = np.array(filw)
    mean = np.mean(element_backg, axis=0)
    sd = np.std(element_backg, axis=0)
    final_list_backg = [x for x in filw if (x > mean - factorSD * sd)]
    final_list_backg = [x for x in final_list_backg if (x < mean + factorSD * sd)]